# CSIS3754 Assignment 4 – 2026
## Unsupervised Machine Learning – Credit Card Customer Clustering

## 1.3.1 — Read dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

credit = pd.read_csv('credit_data.csv')
print(credit.head())

## 1.3.2 — Inspect the data

In [ ]:
# Last 20 records
print(credit.tail(20))

# Concise summary
credit.info()

# Unique values per feature
print(credit.nunique())

## 1.3.3 — Discuss difference between Record_Nr and Customer_ID unique values

In [ ]:
print("""
The difference between the number of unique values of Record_Nr and
Customer_ID indicates that there are duplicate Customer_IDs in the
dataset. Record_Nr is a sequential record counter that increments for
every row, so it will always have as many unique values as there are
rows. Customer_ID however should uniquely identify each customer —
if it has fewer unique values than Record_Nr, it means some customers
appear more than once in the dataset as duplicate records. These
duplicates need to be removed to ensure each customer is only
represented once in the analysis.
""")

## 1.3.4 — Handle duplicate Customer_IDs

In [ ]:
# Proof of the issue
print('Total rows:              ', len(credit))
print('Unique Customer_IDs:     ', credit['Customer_ID'].nunique())
print('Number of duplicates:    ', credit.duplicated(subset='Customer_ID').sum())
print('\nDuplicate rows:')
print(credit[credit.duplicated(subset='Customer_ID', keep=False)].sort_values('Customer_ID').head(10))

In [ ]:
# Rectify: remove duplicate Customer_IDs, keeping first occurrence
credit = credit.drop_duplicates(subset='Customer_ID', keep='first')

# Proof that issue is rectified
print('Total rows after deduplication:', len(credit))
print('Unique Customer_IDs:           ', credit['Customer_ID'].nunique())
print('Remaining duplicates:          ', credit.duplicated(subset='Customer_ID').sum())

## 1.3.5 — Discard Record_Nr and Customer_ID

In [ ]:
credit = credit.drop(columns=['Record_Nr', 'Customer_ID'])
print(credit.columns.tolist())
print(credit.head())

## 1.3.6 — k-Means Clustering

In [ ]:
# Pre-processing: scale features
scaler = StandardScaler()
credit_scaled = scaler.fit_transform(credit)
print('Scaled data shape:', credit_scaled.shape)

In [ ]:
# Elbow curve
inertia = []
k_range = range(2, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(credit_scaled)
    inertia.append(km.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(k_range, inertia, marker='o', color='blue')
plt.title('Elbow Curve')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.xticks(k_range)
plt.tight_layout()
plt.show()

In [ ]:
# Silhouette scores
sil_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(credit_scaled)
    sil_scores.append(silhouette_score(credit_scaled, labels))

plt.figure(figsize=(8, 5))
plt.plot(k_range, sil_scores, marker='o', color='orange')
plt.title('Silhouette Score per k')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Silhouette Score')
plt.xticks(k_range)
plt.tight_layout()
plt.show()

best_k = k_range[sil_scores.index(max(sil_scores))]
print(f'Best k by silhouette score: {best_k} (score = {max(sil_scores):.4f})')

In [ ]:
# Train final model with best k
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
credit['Cluster'] = kmeans.fit_predict(credit_scaled)

# Print cluster labels
print('Cluster labels:')
print(credit['Cluster'].values)

# Print cluster centres (inverse transform to original scale)
centres = scaler.inverse_transform(kmeans.cluster_centers_)
centres_df = pd.DataFrame(centres, columns=credit.columns[:-1])
centres_df.index.name = 'Cluster'
print('\nCluster centres:')
print(centres_df)

## 1.3.7 — Scatter plots with clusters and centroids

In [ ]:
features = credit.columns[:-1].tolist()  # exclude Cluster column
pairs = [(features[i], features[j]) for i in range(len(features)) for j in range(i+1, len(features))]

palette = sns.color_palette('tab10', n_colors=best_k)

for x_feat, y_feat in pairs:
    plt.figure(figsize=(7, 5))
    sns.scatterplot(
        data=credit, x=x_feat, y=y_feat,
        hue='Cluster', palette=palette, alpha=0.6
    )
    # Plot centroids
    for i, row in centres_df.iterrows():
        plt.scatter(row[x_feat], row[y_feat],
                    color=palette[i], marker='X', s=200,
                    edgecolors='black', linewidths=1.5,
                    label=f'Centroid {i}' if i == 0 else '')
    plt.title(f'{x_feat} vs {y_feat}')
    plt.tight_layout()
    plt.show()

## 1.3.8 — Evaluate and discuss optimal k

In [ ]:
print(f"""
=== Evaluation of Optimal k ===

Elbow Curve:
   The elbow curve plots inertia (within-cluster sum of squares) against
   the number of clusters k. The optimal k is where the curve bends sharply
   like an elbow — beyond this point, adding more clusters yields diminishing
   returns in reducing inertia.

Silhouette Score:
   The silhouette score measures how similar each point is to its own cluster
   compared to other clusters. A higher score (closer to 1) indicates better
   defined clusters. The best k from the silhouette score was {best_k}.

Scatter Plots:
   The scatter plots allow visual confirmation of the clusters. If the number
   of visually distinct groupings in the scatter plots matches the optimal k
   suggested by the elbow curve and silhouette score, this confirms that the
   chosen k is appropriate. If the scatter plots show a different number of
   natural groupings, the k value may need to be adjusted accordingly.
   Overall, the combination of all three methods provides a more reliable
   determination of the optimal number of clusters than any single method alone.
""")